In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
show tables in gold;

In [0]:
WITH department_stats AS (
    SELECT
        p.department,
        COUNT(DISTINCT op.order_id)                                            AS orders_with_department,
        COUNT(op.product_id)                                                   AS items
    FROM gold.ic_fact_order_products op
    JOIN gold.ic_dim_product p
        ON op.product_id = p.product_id
    GROUP BY p.department
),
totals AS (
    SELECT COUNT(*)                                                            AS total_orders 
    FROM gold.ic_dim_order
)
SELECT
    ds.department,
    ds.orders_with_department,
    ROUND(ds.orders_with_department * 100.0 / t.total_orders, 2)               AS order_penetration_pct,
    ds.items,
    ROUND(ds.items * 100.0 / SUM(ds.items) OVER (), 2)                         AS item_share_pct
FROM department_stats ds
CROSS JOIN totals t
ORDER BY order_penetration_pct DESC;

In [0]:
SELECT
    p.department,
    COUNT(op.product_id)                                                       AS total_line_items,
    SUM(op.reordered)                                                          AS reordered_items,
    ROUND(AVG(CAST(op.reordered AS DOUBLE)) * 100.0, 2)                        AS reorder_rate_pct
FROM gold.ic_fact_order_products op
JOIN gold.ic_dim_order o
    ON op.order_id = o.order_id
JOIN gold.ic_dim_product p
    ON op.product_id = p.product_id
WHERE NOT o.is_first_order
GROUP BY p.department
ORDER BY reorder_rate_pct DESC;

In [0]:
WITH aisle_performance AS (
    SELECT
        p.department,
        p.aisle,
        COUNT(op.product_id)                                                   AS total_line_items,
        SUM(op.reordered)                                                      AS reordered_items,
        ROUND(AVG(CAST(op.reordered AS DOUBLE)) * 100.0, 2)                    AS reorder_rate_pct
    FROM gold.ic_fact_order_products op
    JOIN gold.ic_dim_order o
        ON op.order_id = o.order_id
    JOIN gold.ic_dim_product p
        ON op.product_id = p.product_id
    WHERE NOT o.is_first_order
    GROUP BY p.department, p.aisle
    HAVING COUNT(op.product_id) >= 10000
),
ranked_aisles AS (
    SELECT
        department,
        aisle,
        total_line_items,
        reorder_rate_pct,
        ROW_NUMBER() OVER (ORDER BY reorder_rate_pct DESC)                     AS rank_desc,
        ROW_NUMBER() OVER (ORDER BY reorder_rate_pct ASC)                      AS rank_asc
    FROM aisle_performance
)
SELECT
    'Top 10 Staple Aisles' AS tier,
    rank_desc AS ranking,
    department,
    aisle,
    total_line_items,
    reorder_rate_pct
FROM ranked_aisles
WHERE rank_desc <= 10

UNION ALL

SELECT
    'Bottom 10 One-Off Aisles' AS tier,
    rank_asc AS ranking,
    department,
    aisle,
    total_line_items,
    reorder_rate_pct
FROM ranked_aisles
WHERE rank_asc <= 10

ORDER BY tier DESC, ranking ASC;

In [0]:
WITH item_basket_context AS (
    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        -- Calculate total items in the basket for this specific order
        COUNT(*) OVER (PARTITION BY order_id)                                  AS basket_size
    FROM gold.ic_fact_order_products
),
normalized_positions AS (
    SELECT
        order_id,
        product_id,
        add_to_cart_order,
        basket_size,
        -- Relative position normalized between 0.0 (first added) and 1.0 (last added)
        -- Using (basket_size - 1) in denominator handles single-item orders safely via NULLIF
        (CAST(add_to_cart_order AS DOUBLE) - 1.0) / NULLIF(CAST(basket_size AS DOUBLE) - 1.0, 0) AS relative_position
    FROM item_basket_context
)
SELECT
    p.department,
    COUNT(np.product_id)                                                       AS total_line_items,
    ROUND(AVG(CAST(np.add_to_cart_order AS DOUBLE)), 2)                        AS avg_cart_position,
    ROUND(AVG(np.relative_position), 4)                                        AS avg_relative_position
FROM normalized_positions np
JOIN gold.ic_dim_product p
    ON np.product_id = p.product_id
GROUP BY p.department
ORDER BY avg_relative_position ASC;

In [0]:
WITH order_sizes AS (
    SELECT
        order_id,
        COUNT(*) AS basket_size
    FROM gold.ic_fact_order_products
    GROUP BY order_id
)
SELECT
    ROUND(AVG(CAST(basket_size AS DOUBLE)), 2) AS avg_basket_size,
    percentile_approx(basket_size, 0.25) AS p25_basket_size,
    percentile_approx(basket_size, 0.50) AS median_basket_size,
    percentile_approx(basket_size, 0.75) AS p75_basket_size,
    percentile_approx(basket_size, 0.90) AS p90_basket_size
FROM order_sizes;

In [0]:
WITH order_sizes AS (
    SELECT
        order_id,
        COUNT(*) AS basket_size
    FROM gold.ic_fact_order_products
    GROUP BY order_id
)
SELECT
    o.order_number,
    COUNT(o.order_id) AS total_orders_at_this_number,
    ROUND(AVG(CAST(os.basket_size AS DOUBLE)), 2) AS avg_basket_size
FROM gold.ic_dim_order o
JOIN order_sizes os
    ON o.order_id = os.order_id
WHERE o.order_number BETWEEN 1 AND 20
GROUP BY o.order_number
ORDER BY o.order_number ASC;

In [0]:
WITH top_100_products AS (
    -- Step 1: Identify the top 100 products by unique order count
    SELECT op.product_id
    FROM gold.ic_fact_order_products op
    GROUP BY op.product_id
    ORDER BY COUNT(DISTINCT op.order_id) DESC
    LIMIT 100
),
order_items AS (
    -- Step 2: Filter fact rows to include only orders containing the top 100 products
    SELECT DISTINCT
        op.order_id,
        op.product_id
    FROM gold.ic_fact_order_products op
    INNER JOIN top_100_products tp
        ON op.product_id = tp.product_id
),
product_order_counts AS (
    -- Get individual order counts for support and confidence denominators (P(A), P(B))
    SELECT 
        product_id,
        COUNT(DISTINCT order_id) AS order_count
    FROM order_items
    GROUP BY product_id
),
total_orders_cte AS (
    -- Total order count baseline (3,346,083)
    SELECT COUNT(*) AS total_orders FROM gold.ic_dim_order
),
pairs AS (
    -- Step 3: Self-join to form unique pairs (A, B) where A < B
    SELECT
        a.product_id AS product_a,
        b.product_id AS product_b,
        COUNT(DISTINCT a.order_id) AS pair_orders
    FROM order_items a
    INNER JOIN order_items b
        ON a.order_id = b.order_id
       AND a.product_id < b.product_id
    GROUP BY a.product_id, b.product_id
    HAVING COUNT(DISTINCT a.order_id) >= 1000
),
metrics_calculation AS (
    SELECT
        p.product_a,
        p.product_b,
        p.pair_orders,
        t.total_orders,
        pa.order_count AS orders_a,
        pb.order_count AS orders_b,
        
        -- Support: P(A and B) = orders with both / total orders
        CAST(p.pair_orders AS DOUBLE) / t.total_orders AS support,
        
        -- Confidence: P(B | A) = orders with both / orders with A
        CAST(p.pair_orders AS DOUBLE) / pa.order_count AS confidence_a_to_b,
        
        -- Lift: P(A and B) / (P(A) * P(B))
        (CAST(p.pair_orders AS DOUBLE) / t.total_orders) / 
        ((CAST(pa.order_count AS DOUBLE) / t.total_orders) * (CAST(pb.order_count AS DOUBLE) / t.total_orders)) AS lift
    FROM pairs p
    CROSS JOIN total_orders_cte t
    INNER JOIN product_order_counts pa ON p.product_a = pa.product_id
    INNER JOIN product_order_counts pb ON p.product_b = pb.product_id
)
SELECT
    m.product_a,
    pa_dim.aisle AS aisle_a,
    pa_dim.department AS dept_a,
    m.product_b,
    pb_dim.aisle AS aisle_b,
    pb_dim.department AS dept_b,
    m.pair_orders,
    ROUND(m.support * 100, 4) AS support_pct,
    ROUND(m.confidence_a_to_b * 100, 2) AS confidence_pct,
    ROUND(m.lift, 2) AS lift
FROM metrics_calculation m
LEFT JOIN gold.ic_dim_product pa_dim ON m.product_a = pa_dim.product_id
LEFT JOIN gold.ic_dim_product pb_dim ON m.product_b = pb_dim.product_id
ORDER BY lift DESC
LIMIT 20;